# Qwen 3.5 LiteRT Kaggle Setup

This notebook converts the Qwen 3.5 model to LiteRT (`.tflite`) format on **Kaggle**.

Kaggle provides **30GB RAM** — enough for full conversion with all prefill sizes.

In [ ]:
# 1. Clone the repository and install dependencies
import os

WORK_DIR = '/kaggle/working/litert-torch-qwen3.5'

if not os.path.exists(WORK_DIR):
    !git clone https://github.com/byepv2493k-ship-it/litert-torch-qwen3.5.git {WORK_DIR}

%cd {WORK_DIR}

# Install core LiteRT / AI Edge packages
!pip install ai-edge-litert-nightly==2.2.0.dev20260319 ai-edge-quantizer-nightly==0.5.1.dev20260320

# Install torchao nightly (needs pt2e submodule used by litert_torch)
!pip install --extra-index-url https://download.pytorch.org/whl/nightly/cpu torchao==0.16.0

# Install litert_torch from this repo
!pip install -e .

# Install TensorFlow + PyTorch nightly
!pip install tensorflow==2.21.0
!pip install --extra-index-url https://download.pytorch.org/whl/nightly/cpu torch==2.12.0.dev20260320+cpu

# Install HuggingFace + other deps
!pip install huggingface_hub==1.7.2 transformers==5.3.0 safetensors==0.7.0 sentencepiece==0.2.1 tokenizers==0.22.2
!pip install numpy==2.4.3 scipy==1.17.1 jax==0.9.2 jaxlib==0.9.2 ml_dtypes==0.5.4
!pip install absl-py fire flatbuffers protobuf PyYAML Jinja2 sympy tqdm regex pillow

In [ ]:
# 2. Download the Qwen Model
import os
from huggingface_hub import snapshot_download

# Option A: Set token directly
os.environ['HF_TOKEN'] = 'put_your_hf_token_here'

# Option B: Use Kaggle Secrets (recommended)
# Go to Add-ons -> Secrets -> Add 'HF_TOKEN' with your token
# Then uncomment:
# from kaggle_secrets import UserSecretsClient
# os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

model_path = snapshot_download(
    'Qwen/Qwen3.5-0.8B',
    token=os.environ['HF_TOKEN'], 
    ignore_patterns=['*.bin','*.msgpack','*.onnx']
)
print(f'Model downloaded to: {model_path}')

In [ ]:
# Fix ai_edge_quantizer / ai_edge_litert version mismatch
import ai_edge_litert.tools.mmap_utils as _mu
if not hasattr(_mu, "advise_dont_need"):
    _mu.advise_dont_need = lambda *a, **kw: None
    print("Patched mmap_utils.advise_dont_need")
else:
    print("mmap_utils.advise_dont_need already exists")

In [ ]:
# 3. Run the Converter Script
# With 30GB RAM on Kaggle, you can use larger values than on Colab.
# Adjust kv_cache_max_len and prefill_seq_lens as needed.
OUTPUT_DIR = '/kaggle/working/output'
!mkdir -p {OUTPUT_DIR}

!python -m litert_torch.generative.examples.qwen.convert_v3_5_to_tflite \
    --checkpoint_path={model_path} \
    --output_path={OUTPUT_DIR} \
    --model_size=0.8b \
    --quantize=dynamic_int8 \
    --kv_cache_max_len=2048 \
    --prefill_seq_lens=8 \\
    --prefill_seq_lens=64 \\
    --prefill_seq_lens=128 \\
    --prefill_seq_lens=256 \\
    --prefill_seq_lens=512

print("Conversion finished. Check the output directory.")

In [ ]:
# 4. Check output files
# On Kaggle, files in /kaggle/working/ are automatically available
# in the 'Output' tab after the notebook finishes running.
import os
import glob

output_files = glob.glob(os.path.join(OUTPUT_DIR, '*'))
print(f"Found {len(output_files)} file(s):")
for f in output_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  - {os.path.basename(f)} ({size_mb:.1f} MB)")

print(f"\nFiles are in: {OUTPUT_DIR}")
print("After the notebook finishes, go to the 'Output' tab to download them.")